<div align="center">

# Week 09: Energy Optimisation: Economic Dispatch, Battery Scheduling and Demand Response

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-09) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/Week09_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also holds the day of Week 1, the battery, the tariff of the campus, the three units of the regional grid and the ten cars of the car park. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog, minimize_scalar

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

# The spring weekday of Week 1 on the campus, one value per hour (kW and m/s)
HOURS = np.arange(24)
LOAD_KW = np.array([262, 255, 250, 248, 252, 270, 330, 430, 520, 570, 585, 590,
                    575, 580, 585, 575, 560, 530, 520, 500, 450, 390, 320, 280], float)
EV_KW = np.array([0, 0, 0, 0, 0, 0, 0, 22, 66, 66, 44, 22,
                  22, 22, 22, 44, 66, 110, 110, 88, 44, 22, 0, 0], float)
WIND_MS = np.array([7.8, 8.0, 8.2, 8.0, 7.6, 7.0, 6.2, 5.4, 4.8, 4.4, 4.2, 4.6,
                    5.0, 5.2, 5.0, 4.6, 4.2, 3.8, 3.6, 3.5, 3.8, 4.6, 5.8, 7.0])
G_SKY = 950 * np.maximum(0, np.sin(np.pi * (HOURS + 0.5 - 6) / 13))
T_AMB = 12 + 7 * np.sin(np.pi * (HOURS - 9) / 12)

def net_demand(cloud=1.0, ev=EV_KW):
    """The import of the campus without the battery (kW): load and cars minus the solar plant and the wind turbine."""
    t_cell = T_AMB + 0.03 * cloud * G_SKY
    pv = np.round(np.maximum(0.0, 400 * cloud * G_SKY / 1000 * (1 - 0.004 * (t_cell - 25))), 1)
    p = 250 * (WIND_MS ** 3 - 27) / (12 ** 3 - 27)
    wind = np.round(np.where(WIND_MS < 3, 0.0, np.where(WIND_MS < 12, p, 250.0)), 1)
    return LOAD_KW + ev - pv - wind

NET_CLEAR, NET_CLOUDY = net_demand(1.0), net_demand(0.4)
# The battery (Week 2) and the tariff of the campus
E_KWH, E_MIN, E_MAX, E0, P_BAT, ETA, IMPORT_MAX = 500.0, 50.0, 450.0, 250.0, 250.0, 0.95, 600.0
S_EXPORT, K_PEAK, V_STORED = 0.05, 0.30, 0.15       # euro/kWh exported, euro/kW of the peak, euro/kWh stored
# The three units of the regional grid: cost b P + c P^2 (euro/h, P in MW) and their limits (MW)
B_UNIT, C_UNIT = np.array([20.0, 35.0, 60.0]), np.array([0.02, 0.025, 0.25])
P_MIN, P_MAX = np.array([100.0, 0.0, 0.0]), np.array([600.0, 500.0, 300.0])
D_REGION = np.array([600, 570, 550, 545, 550, 580, 680, 820, 950, 1020, 1050, 1060,
                     1040, 1030, 1020, 1010, 1040, 1120, 1220, 1260, 1200, 1050, 850, 700], float)
# The ten cars of the car park: name, arrival hour, departure hour, energy (kWh), at most 22 kW each
CARS = [("A", 7, 17, 110), ("B", 8, 17, 66), ("C", 8, 13, 44), ("D", 12, 20, 110), ("E", 15, 19, 66),
        ("F", 16, 24, 88), ("G", 17, 24, 88), ("H", 17, 21, 44), ("J", 17, 24, 110), ("K", 18, 22, 44)]
print("Setup complete. Running in Colab:", IN_COLAB, "| peak import without the battery:", NET_CLEAR.max(),
      "kW on the clear day,", NET_CLOUDY.max(), "kW on the cloudy day")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 9.1": (lambda: 500.0, 1e-4),
    "Exercise 9.2": (lambda: 0.24 * 190, 1e-6),
    "Exercise 9.3": (lambda: 240.0, 1e-6),
    "Exercise A": (lambda: two_hours(300, 470)['saving'], 1e-6),
    "Exercise B": (lambda: dispatch(900)[0], 1e-4),
    "Exercise C": (lambda: day_cost(NET_CLOUDY) - schedule(NET_CLOUDY)['cost'], 1e-4),
    "Exercise D": (lambda: 88, 1e-9),
    "Exercise E": (lambda: STAGE9['clear day: saving of the optimal battery with smart charging (percent)'], 1e-6),
}

## Python step 9: Optimisation with SciPy

Optimisation finds the best decision within its limits. This step minimises a function of one variable, solves a linear program in two variables with `linprog` and plans a battery over a few hours with arrays &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#ref-3)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "CARS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-09/"
               "NB09_energy_optimisation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### An objective function and its minimum

In [ ]:
# Python practice: a small example for learning the language
lam = 50.0                                          # the price of energy in euro per MWh
loss = lambda P: 35 * P + 0.025 * P ** 2 - lam * P  # the cost minus the revenue of a unit in one hour
best = minimize_scalar(loss, bounds=(0, 500), method="bounded")
print(f"the unit produces {best.x:.1f} MW and gains {-best.fun:.0f} euro in the hour")

`minimize_scalar` searches the minimum of a function of one variable within its bounds. The result keeps the best value in the attribute `x` and the minimum itself in `fun`. Here the unit produces until its incremental cost $35 + 0.05P$ reaches the price. The output $P$ is in megawatts (MW) and the price in euro per megawatt hour (MWh).

### A linear program in two variables

In [ ]:
# Python practice: a small example for learning the language
c = [-0.22, -0.24]               # linprog minimises, so the saving per kWh enters with a minus sign
A_ub, b_ub = [[1, 1]], [285]     # both hours together: 300 kWh in the battery deliver 285 kWh
res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=[(0, 250), (0, 250)], method="highs")
print("discharge at 18:00 and 19:00 (kW):", res.x, "| saving:", round(-res.fun, 2), "euro")

`linprog` minimises $c^T x$ subject to $A_{ub}\,x \le b_{ub}$ and to bounds for each variable. The method `highs` is a modern simplex and interior-point solver. The optimum gives the dearer hour as much as it can take and the rest to the other hour.

### Variables for every hour

In [ ]:
# Python practice: a small example for learning the language
price = np.array([0.15, 0.14, 0.22, 0.24])         # four hours (euro/kWh)
plan = np.array([-100, -100, 90, 100])               # battery power: charging negative, discharging positive (kW)
energy = E0 - np.cumsum(np.where(plan > 0, plan / ETA, plan * ETA))
print("stored energy after each hour (kWh):", np.round(energy, 2))
print("saving of the plan (euro):", round(float(price @ plan), 2))

`np.where` chooses element by element: Losses on the way out for a discharge and on the way in for a charge. The function `np.cumsum` adds up the changes hour by hour, and `price @ plan` is the sum of the products, the saving of the plan.

**Quick check.** Why does the plan save money although the battery loses 5 percent in each direction?

<details><summary>Answer</summary>

It buys at 0.14 and 0.15 euro per kWh and avoids imports at 0.22 and 0.24 euro per kWh: The difference in price is larger than the losses.

</details>

### Exercises for Python step 9

**Exercise 9.1.** Minimise the cost minus the revenue of a unit with the cost $20P + 0.02P^2$ at a price of 40 euro per MWh. Keep the output within 100 and 600 MW and store it in `p_91`.

In [ ]:
p_91 = None   # your answer

check("Exercise 9.1", p_91)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_91 = minimize_scalar(lambda P: 20 * P + 0.02 * P ** 2 - 40 * P, bounds=(100, 600),
                       method="bounded").x

check("Exercise 9.1", p_91)

**Exercise 9.2.** Solve the linear program again when the battery holds only 200 kWh, which deliver 190 kWh, and store the saving in `s_92`.

In [ ]:
s_92 = None   # your answer

check("Exercise 9.2", s_92)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
s_92 = -linprog(c, A_ub=[[1, 1]], b_ub=[190], bounds=[(0, 250), (0, 250)],
                method="highs").fun

check("Exercise 9.2", s_92)

**Exercise 9.3.** Store the lowest stored energy of `energy` in `e_93`.

In [ ]:
e_93 = None   # your answer

check("Exercise 9.3", e_93)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
e_93 = float(energy.min())

check("Exercise 9.3", e_93)

## 1. Optimisation: The two evening hours &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#ref-1)&#93;

*Lecture: [Optimisation: Variables, objective and constraints](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#optimisation)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "energy" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-09/"
               "NB09_energy_optimisation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Every optimisation problem has decision variables, an objective and constraints. Python writes the two evening hours of the battery as a linear program, solves it and draws the feasible region with the optimum.

**From the formulas to the code.** The program maximises $p_{18} d_{18} + p_{19} d_{19}$ subject to $d_{18} + d_{19} \le E\eta$, $d_t \ge n_t - G$ and $0 \le d_t \le 250$, in the function `two_hours(E, G)`. `linprog` minimises, so the code negates the objective, and the peak constraint enters as $-d_t \le G - n_t$.

In [ ]:
P18, P19 = 0.22, 0.24                 # the prices at 18:00 and 19:00 (euro/kWh), from Section 2
N18, N19 = NET_CLEAR[18], NET_CLEAR[19]  # the import without the battery (kW)

def two_hours(E_avail=300.0, G=600.0):
    """The best discharge at 18:00 and 19:00 with E_avail kWh above the minimum and an import of at most G kW."""
    res = linprog([-P18, -P19], A_ub=[[1, 1], [-1, 0], [0, -1]], b_ub=[E_avail * ETA, G - N18, G - N19],
                  bounds=[(0, P_BAT), (0, P_BAT)], method="highs")
    if res.status != 0:
        return None                                  # no plan meets all constraints
    return dict(d18=res.x[0], d19=res.x[1], saving=-res.fun)

for E_avail, G in [(300, 600), (300, 450), (250, 480), (150, 450)]:
    r = two_hours(E_avail, G)
    print(f"{E_avail} kWh stored, peak at most {G} kW: " + ("infeasible" if r is None else
          f"discharge {r['d18']:.1f} and {r['d19']:.1f} kW, saving {r['saving']:.2f} euro"))

**Python note.** `res.status` is 0 when the solver found the optimum and 2 when the constraints contradict each other. The function then returns `None`, which the loop prints as infeasible.

**Exercise A.** Store the saving of the two hours with 300 kWh stored and an import of at most 470 kW in `s_a`.

In [ ]:
s_a = None   # your answer

check("Exercise A", s_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
s_a = two_hours(300, 470)['saving']

check("Exercise A", s_a)

In [ ]:
# Figure: the feasible region of the two evening hours and the optimum
E_avail, G = 300, 450
d = np.linspace(0, 260, 521)
X18, X19 = np.meshgrid(d, d)
ok = (X18 + X19 <= E_avail * ETA) & (X18 >= N18 - G) & (X19 >= N19 - G) & (X18 <= 250) & (X19 <= 250)
r = two_hours(E_avail, G)
fig, ax = plt.subplots(figsize=(5.6, 4.4))
ax.contourf(X18, X19, ok, levels=[0.5, 1.5], colors=["#DCEBF5"])
for s in (40, 55, r["saving"]):
    ax.plot(d, (s - P18 * d) / P19, ls=":" if s != r["saving"] else "-", color="#C98510", lw=1)
ax.plot(r["d18"], r["d19"], "o", color="#B53F2E", ms=8, label=f"optimum, saving {r['saving']:.2f} euro")
ax.set_xlim(0, 260)
ax.set_ylim(0, 260)
ax.set_xlabel("discharge at 18:00 (kW)")
ax.set_ylabel("discharge at 19:00 (kW)")
ax.legend(fontsize=8)
plt.show()

**Observe and reflect.** Why does the optimum always lie at a corner of the feasible region? What happens to the region when the battery holds less energy or the peak limit falls?

## 2. Economic dispatch: The price of each hour &#91;[2](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#ref-2)&#93;

*Lecture: [Economic dispatch](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#dispatch)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "two_hours" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-09/"
               "NB09_energy_optimisation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The regional grid serves its demand with three units of different costs. Python finds the output of each unit at the lowest total cost. Its incremental cost $\lambda$ sets the price of energy for the campus in every hour.

**From the formulas to the code.** With $C_i(P_i) = b_i P_i + c_i P_i^2$, the optimum gives every unit within its limits the same incremental cost $\lambda = b_i + 2c_iP_i$. `dispatch(D)` searches $\lambda$ by bisection until the outputs $P_i = (\lambda - b_i)/(2c_i)$, clipped to the limits by `outputs`, add up to $D$. The price is $p_t = 0.10 + \lambda_t/1000$ euro per kWh.

In [ ]:
def outputs(lam):
    """The output of each unit in MW at the incremental cost lam (euro/MWh), within its limits."""
    return np.clip((lam - B_UNIT) / (2 * C_UNIT), P_MIN, P_MAX)

def dispatch(D, tol=1e-7):
    """The economic dispatch of the demand D (MW): the incremental cost lam and the outputs of the three units."""
    lo, hi = 0.0, 1000.0
    while hi - lo > tol:
        lam = (lo + hi) / 2
        if outputs(lam).sum() < D:
            lo = lam
        else:
            hi = lam
    lam = (lo + hi) / 2
    return lam, outputs(lam)

for D in (550, 1060, 1260):
    lam, P = dispatch(D)
    print(f"demand {D} MW: lambda {lam:.2f} euro/MWh, outputs {np.round(P, 1)} MW, "
          f"cost {np.sum(B_UNIT * P + C_UNIT * P ** 2):,.0f} euro/h")
LAMBDA = np.array([dispatch(D)[0] for D in D_REGION])
PRICE = np.round(0.10 + LAMBDA / 1000, 4)
print("price of the campus by hour (euro/kWh):")
print(PRICE)

**Python note.** Bisection halves the interval of $\lambda$ until it is shorter than `tol`: The total output grows with $\lambda$, so the demand lies in one half. `np.clip` holds every unit within its limits.

**Exercise B.** Store the incremental cost $\lambda$ for a regional demand of 900 MW in `lam_b`.

In [ ]:
lam_b = None   # your answer

check("Exercise B", lam_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
lam_b = dispatch(900)[0]

check("Exercise B", lam_b)

In [ ]:
# Figure: the supply curve of the region and the price of the campus over the day
lams = np.linspace(20, 200, 721)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot([outputs(l).sum() for l in lams], lams, color="#1E6FA8")
for D, col in ((550, "#2E8B57"), (1060, "#C98510"), (1260, "#B53F2E")):
    ax[0].axvline(D, color=col, ls="--", lw=1, label=f"{D} MW: {dispatch(D)[0]:.1f} euro/MWh")
ax[0].set_xlabel("total output (MW)")
ax[0].set_ylabel("incremental cost (euro/MWh)")
ax[0].legend(fontsize=7)
ax[1].bar(HOURS + 0.5, PRICE, width=0.9, color="#1E6FA8")
ax[1].set_ylim(0.12, 0.25)
ax[1].set_xlabel("hour")
ax[1].set_ylabel("price of the campus (euro/kWh)")
plt.tight_layout()
plt.show()

**Observe and reflect.** Why is the price almost flat at night and steep in the evening? Which unit sets the price at 19:00?

## 3. Battery scheduling: Your runs at the battery container &#91;[5](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#ref-5)&#93;

*Lecture: [Battery scheduling](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#battery-schedule)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "PRICE" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-09/"
               "NB09_energy_optimisation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** With the prices of Section 2, Python writes the battery schedule of a whole day as one linear program. It compares the optimum with the rule of Week 1 and with a rule of prices, and it checks your runs at the battery container.

**From the formulas to the code.** The objective $\sum_t (p_t g_t - s\,x_t) + K\,G$ with the balance $g_t - x_t = n_t + c_t - d_t$, the storage $e_t = e_{t-1} + \eta c_t - d_t/\eta$ and the peak $g_t \le G$ is built in `schedule(net)`. `day_cost` prices any plan, and `corrected` adds 0.15 euro for every kWh missing in the battery at midnight.

In [ ]:
def day_cost(grid):
    """The cost of a day in euro: the energy at the hourly price, the export at 0.05 euro/kWh and the peak charge."""
    grid = np.asarray(grid, float)
    return float(PRICE @ grid.clip(min=0) - S_EXPORT * (-grid).clip(min=0).sum() + K_PEAK * max(0.0, grid.max()))

def corrected(cost, e_end):
    """The cost with the energy missing in the battery at midnight priced at 0.15 euro/kWh."""
    return cost + V_STORED * (E0 - e_end)

def schedule(net, cars=None, battery=True):
    """The cheapest day as a linear program. net: the import without the battery (kW); cars: flexible charging."""
    T = 24
    ev = [(k, t) for k, (_, a, dep, _) in enumerate(cars or []) for t in range(a, dep)]
    n = 5 * T + 1 + len(ev)                        # c, d, g, x, e for every hour, the peak G, the charging of cars
    C, D, Gr, X, E, PK = 0, T, 2 * T, 3 * T, 4 * T, 5 * T
    cost = np.zeros(n)
    cost[Gr:Gr + T], cost[X:X + T], cost[PK] = PRICE, -S_EXPORT, K_PEAK
    A_eq, b_eq, A_ub, b_ub = [], [], [], []
    for t in range(T):
        row = np.zeros(n)                          # the balance: g - x - c + d - cars = net
        row[[Gr + t, X + t, C + t, D + t]] = [1, -1, -1, 1]
        row[[PK + 1 + j for j, (k, h) in enumerate(ev) if h == t]] = -1
        A_eq.append(row)
        b_eq.append(net[t])
        row = np.zeros(n)                          # the storage: e_t - e_(t-1) - eta c_t + d_t / eta = 0
        row[[E + t, C + t, D + t]] = [1, -ETA, 1 / ETA]
        if t > 0:
            row[E + t - 1] = -1
        A_eq.append(row)
        b_eq.append(E0 if t == 0 else 0.0)
        row = np.zeros(n)                          # the peak: g_t - G <= 0
        row[[Gr + t, PK]] = [1, -1]
        A_ub.append(row)
        b_ub.append(0.0)
    for k, (_, a, dep, energy) in enumerate(cars or []):
        row = np.zeros(n)                          # every car gets its energy before it leaves
        row[[PK + 1 + j for j, (kk, h) in enumerate(ev) if kk == k]] = 1
        A_eq.append(row)
        b_eq.append(energy)
    row = np.zeros(n)
    row[E + T - 1] = -1                            # the day ends with at least the stored energy of the start
    A_ub.append(row)
    b_ub.append(-E0)
    p_bat = P_BAT if battery else 0.0
    bounds = ([(0, p_bat)] * 2 * T + [(0, IMPORT_MAX)] * T + [(0, None)] * T + [(E_MIN, E_MAX)] * T + [(0, None)]
              + [(0, 22)] * len(ev))
    res = linprog(cost, A_ub=np.array(A_ub), b_ub=b_ub, A_eq=np.array(A_eq), b_eq=b_eq, bounds=bounds,
                  method="highs")
    x = res.x
    ev_car = np.zeros((len(cars or []), T))       # the charging of every car in every hour
    for j, (k, t) in enumerate(ev):
        ev_car[k, t] = x[PK + 1 + j]
    return dict(cost=res.fun, bat_kw=x[D:D + T] - x[C:C + T], grid_kw=x[Gr:Gr + T] - x[X:X + T],
                soc=x[E:E + T] / E_KWH, peak=x[PK], ev_kw=ev_car.sum(axis=0), ev_car=ev_car)

def run_rule(net, rule):
    """Runs a battery rule hour by hour; rule(t, n_t, e) returns the power of the battery (+ discharge, - charge)."""
    e, grid, bat, soc = E0, [], [], []
    for t, n_t in enumerate(net):
        p = rule(t, n_t, e)
        e += -p / ETA if p > 0 else -p * ETA
        grid.append(n_t - p)
        bat.append(p)
        soc.append(e / E_KWH)
    return dict(grid_kw=np.array(grid), bat_kw=np.array(bat), soc=np.array(soc), e_end=e)

def peak_rule(threshold=450.0, charge_below=330.0):
    """The rule of Week 1: discharge above the threshold, charge below charge_below."""
    def rule(t, n_t, e):
        if n_t > threshold:
            return min(n_t - threshold, P_BAT, (e - E_MIN) * ETA)
        if n_t < charge_below:
            return -min(charge_below - n_t, P_BAT, (E_MAX - e) / ETA)
        return 0.0
    return rule

def price_rule(low, high):
    """Charge at full power when the price is at most low, discharge when it is at least high, never export."""
    def rule(t, n_t, e):
        if PRICE[t] >= high:
            return min(P_BAT, max(0.0, n_t), (e - E_MIN) * ETA)
        if PRICE[t] <= low:
            return -min(P_BAT, (E_MAX - e) / ETA, IMPORT_MAX - n_t)
        return 0.0
    return rule

for name, net in (("clear", NET_CLEAR), ("cloudy", NET_CLOUDY)):
    rule = run_rule(net, peak_rule(450))
    best = schedule(net)
    print(f"{name} day: without the battery {day_cost(net):.2f} euro, peak {net.max():.1f} kW")
    print(f"  rule of Week 1: {day_cost(rule['grid_kw']):.2f} euro, corrected "
          f"{corrected(day_cost(rule['grid_kw']), rule['e_end']):.2f} euro, peak {rule['grid_kw'].max():.1f} kW")
    print(f"  optimal schedule: {best['cost']:.2f} euro, peak {best['peak']:.1f} kW")

**Python note.** `row[[...]] = [...]` sets several elements of a row at once. Each constraint is one row of the matrix `A_eq` or `A_ub`, and the 121 variables of the day stand side by side in one vector.

**Exercise C.** Store the saving of the optimal schedule against the day without the battery on the cloudy day in `s_c`.

In [ ]:
s_c = None   # your answer

check("Exercise C", s_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
s_c = day_cost(NET_CLOUDY) - schedule(NET_CLOUDY)['cost']

check("Exercise C", s_c)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lab.html) runs these strategies at the battery container. Paste the copied rows into `MY_RUNS`, and the code runs each of your days again.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_RUNS = """
"""
if MY_RUNS.strip():
    runs = pd.read_csv(io.StringIO(MY_RUNS.strip()))
    source = "your own runs at the battery container"
else:
    runs = pd.DataFrame({"run": [1, 2, 3, 4, 5, 6], "day": ["clear", "clear", "clear", "cloudy", "cloudy", "clear"],
                         "strategy": ["none", "peak rule", "optimal", "peak rule", "optimal", "price rule"],
                         "threshold_kw": [None, 450, None, 450, None, None], "price_low": [None] * 5 + [0.14],
                         "price_high": [None] * 5 + [0.2], "corrected": [1446.09, 1382.98, 1379.91, 1723.09,
                                                                         1669.91, 1394.53]})
    source = "a simulated student"
def model_cost(r):
    net = NET_CLEAR if r.day == "clear" else NET_CLOUDY
    if r.strategy == "none":
        return day_cost(net)
    if r.strategy == "optimal":
        return schedule(net)["cost"]
    rule = peak_rule(r.threshold_kw) if r.strategy == "peak rule" else price_rule(r.price_low, r.price_high)
    out = run_rule(net, rule)
    return corrected(day_cost(out["grid_kw"]), out["e_end"])
runs["model"] = [round(model_cost(r), 2) for r in runs.itertuples()]
print("Data:", source)
print(runs[["run", "day", "strategy", "threshold_kw", "price_low", "price_high", "corrected", "model"]]
      .to_string(index=False))

In [ ]:
# Figure: the optimal schedule of the clear day against the rule of Week 1
best, rule = schedule(NET_CLEAR), run_rule(NET_CLEAR, peak_rule(450))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].step(HOURS, NET_CLEAR, where="post", color="#56616D", label="without the battery")
ax[0].step(HOURS, rule["grid_kw"], where="post", color="#C98510", label="rule of Week 1")
ax[0].step(HOURS, best["grid_kw"], where="post", color="#1E6FA8", lw=2, label="optimal schedule")
ax[0].set_xlabel("hour")
ax[0].set_ylabel("import (kW)")
ax[0].legend(fontsize=8)
ax[1].plot(HOURS + 1, rule["soc"], color="#C98510", label="rule of Week 1")
ax[1].plot(HOURS + 1, best["soc"], color="#1E6FA8", lw=2, label="optimal schedule")
ax[1].set_xlabel("hour")
ax[1].set_ylabel("state of charge")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Observe and reflect.** On which day does the rule of Week 1 lose most against the optimum, and why? What does the optimum do that a fixed threshold cannot?

## 4. Demand response: Your runs in the car park &#91;[6](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#ref-6)&#93;

*Lecture: [Demand response](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#demand-response)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "schedule" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-09/"
               "NB09_energy_optimisation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Ten cars charge in the car park, each between its arrival and its departure. Python compares charging at once, in the cheapest hours and under a limit of power. It then lets the linear program of Section 3 shift the charging as well.

**From the formulas to the code.** Car $k$ needs $E_k$ kWh between its arrival $a_k$ and its departure $d_k$, with at most 22 kW: $\sum_{t=a_k}^{d_k-1} P_{k,t} = E_k$. `ev_immediate`, `ev_cheapest` and `ev_limited(cap)` build the power of the car park. The last one serves the car that leaves first, as earliest deadline first (EDF) does in Week 7.

In [ ]:
BASE_CLEAR = NET_CLEAR - EV_KW                   # the campus without the car park

def ev_immediate():
    """Every car charges at 22 kW from its arrival until it is full."""
    p = np.zeros(24)
    for name, a, dep, energy in CARS:
        left, t = energy, a
        while left > 1e-9:
            q = min(22.0, left)
            p[t] += q
            left -= q
            t += 1
    return p

def ev_cheapest():
    """Every car charges in the cheapest hours between its arrival and its departure."""
    p = np.zeros(24)
    for name, a, dep, energy in CARS:
        left = energy
        for t in sorted(range(a, dep), key=lambda h: (PRICE[h], h)):
            q = min(22.0, left)
            p[t] += q
            left -= q
            if left <= 1e-9:
                break
    return p

def ev_limited(cap):
    """At most cap kW for the car park, the car that leaves first served first; also returns the energy missing."""
    left = {name: energy for name, a, dep, energy in CARS}
    p = np.zeros(24)
    for t in range(24):
        room = cap
        for name, a, dep, energy in sorted(CARS, key=lambda c: (c[2], c[0])):
            if a <= t < dep and left[name] > 1e-9 and room > 1e-9:
                q = min(22.0, left[name], room)
                p[t] += q
                left[name] -= q
                room -= q
    return p, sum(left.values())

print("charging at once equals the car park of Week 1:", np.allclose(ev_immediate(), EV_KW))
for name, ev in (("at once", ev_immediate()), ("cheapest hours", ev_cheapest()), ("limit 88 kW", ev_limited(88)[0])):
    grid = BASE_CLEAR + ev
    print(f"{name:<15} cost of the campus without the battery {day_cost(grid):.2f} euro, peak {grid.max():.1f} kW")
cap_min = next(c for c in range(22, 221, 11) if ev_limited(c)[1] <= 1e-6)
print("smallest limit that charges every car in time:", cap_min, "kW")
dr = schedule(BASE_CLEAR, CARS, battery=False)
print(f"optimal charging without the battery: {dr['cost']:.2f} euro, peak {dr['peak']:.1f} kW")

**Python note.** `sorted(CARS, key=lambda c: (c[2], c[0]))` orders the cars by departure and then by name. A dictionary keeps the energy that each car still needs, and `next(...)` finds the first limit that leaves none.

**Exercise D.** Store the smallest limit of the car park, in kW and in steps of 11 kW, at which every car is charged before it leaves, in `cap_d`.

In [ ]:
cap_d = None   # your answer

check("Exercise D", cap_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
cap_d = next(c for c in range(22, 221, 11) if ev_limited(c)[1] <= 1e-6)

check("Exercise D", cap_d)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lab.html) opens the car park with its ten chargers. Paste the copied rows into `MY_CARS`, and the code runs each of your strategies again.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_CARS = """
"""
if MY_CARS.strip():
    cars = pd.read_csv(io.StringIO(MY_CARS.strip()))
    source = "your own runs in the car park"
else:
    cars = pd.DataFrame({"run": [1, 2, 3, 4, 5], "strategy": ["at once", "cheapest", "limit", "limit", "optimal"],
                         "cap_kw": [None, None, 77, 88, None], "cost": [1446.09, 1408.90, 1437.12, 1443.89,
                                                                       1404.78]})
    source = "a simulated student"
def model_cars(r):
    if r.strategy == "optimal":
        return schedule(BASE_CLEAR, CARS, battery=False)["cost"]
    ev = {"at once": ev_immediate, "cheapest": ev_cheapest}.get(r.strategy, lambda: ev_limited(r.cap_kw)[0])()
    return day_cost(BASE_CLEAR + ev)
cars["model"] = [round(model_cars(r), 2) for r in cars.itertuples()]
print("Data:", source)
print(cars[["run", "strategy", "cap_kw", "cost", "model"]].to_string(index=False))

In [ ]:
# Figure: the car park at once, in the cheapest hours and under the limit, with the price
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for name, ev, col in (("at once", ev_immediate(), "#56616D"), ("cheapest hours", ev_cheapest(), "#1E6FA8"),
                      ("limit 88 kW", ev_limited(88)[0], "#C98510")):
    ax[0].step(HOURS, ev, where="post", color=col, label=name)
ax[0].set_xlabel("hour")
ax[0].set_ylabel("car park (kW)")
ax[0].legend(fontsize=8)
ax[1].step(HOURS, BASE_CLEAR + ev_immediate(), where="post", color="#56616D", label="at once")
ax[1].step(HOURS, BASE_CLEAR + dr["ev_kw"], where="post", color="#1E6FA8", lw=2, label="optimal charging")
ax[1].set_xlabel("hour")
ax[1].set_ylabel("import of the campus (kW)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Observe and reflect.** Why does charging in the cheapest hours raise the import at 23:00? Which cars cannot move at all, and why?

## 5. The campus microgrid, stage 9

*Lecture: [The campus microgrid, stage 9: The dispatch of the microgrid](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#the-campus-microgrid-stage-9)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "ev_limited" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-09/"
               "NB09_energy_optimisation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 9 plans the day of the campus at the lowest cost. The price follows the regional dispatch, the battery follows the optimal schedule and the car park shifts its charging, on a clear and on a cloudy day.

In [ ]:
STAGE9 = {}
STAGE9["price: lowest and highest (euro/kWh)"] = (float(PRICE.min()), float(PRICE.max()))
for name, net in (("clear day", NET_CLEAR), ("cloudy day", NET_CLOUDY)):
    base = net - EV_KW
    none = day_cost(net)
    rule = run_rule(net, peak_rule(450))
    best = schedule(net)
    both = schedule(base, CARS)
    STAGE9[f"{name}: without the battery (euro)"] = round(none, 2)
    STAGE9[f"{name}: rule of Week 1, corrected (euro)"] = round(corrected(day_cost(rule["grid_kw"]), rule["e_end"]), 2)
    STAGE9[f"{name}: optimal battery (euro)"] = round(best["cost"], 2)
    STAGE9[f"{name}: optimal battery with smart charging (euro)"] = round(both["cost"], 2)
    STAGE9[f"{name}: peak with the battery and smart charging (kW)"] = round(both["peak"], 1)
    STAGE9[f"{name}: saving of the optimal battery with smart charging (percent)"] = round(100 * (1 - both["cost"] / none), 2)
STAGE9["smallest limit of the car park (kW)"] = cap_min
for key, value in STAGE9.items():
    print(f"{key:<72} {value}")

**Python note.** A tuple such as `(lowest, highest)` keeps two values under one key of the dictionary.

In [ ]:
# Figure: the costs of the strategies on both days and the plan of the clear day
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
labels = ["without battery", "rule of Week 1", "optimal battery", "battery and cars"]
for shift, name, col in ((-0.2, "clear day", "#F2B33D"), (0.2, "cloudy day", "#56616D")):
    vals = [STAGE9[f"{name}: without the battery (euro)"], STAGE9[f"{name}: rule of Week 1, corrected (euro)"],
            STAGE9[f"{name}: optimal battery (euro)"], STAGE9[f"{name}: optimal battery with smart charging (euro)"]]
    ax[0].bar(np.arange(4) + shift, vals, 0.38, color=col, label=name)
ax[0].set_xticks(range(4))
ax[0].set_xticklabels(labels, fontsize=7)
ax[0].set_ylim(1200, 1800)
ax[0].set_ylabel("cost of the day (euro)")
ax[0].legend(fontsize=8)
both = schedule(NET_CLEAR - EV_KW, CARS)
ax[1].step(HOURS, NET_CLEAR, where="post", color="#56616D", label="without the battery")
ax[1].step(HOURS, both["grid_kw"], where="post", color="#1E6FA8", lw=2, label="battery and cars, optimal")
ax[1].set_xlabel("hour")
ax[1].set_ylabel("import (kW)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Exercise E.** Store the saving in percent of the optimal battery with smart charging on the clear day, as kept in `STAGE9`, in `pct_e`.

In [ ]:
pct_e = None   # your answer

check("Exercise E", pct_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
pct_e = STAGE9['clear day: saving of the optimal battery with smart charging (percent)']

check("Exercise E", pct_e)

**Observe and reflect.** The rule of Week 1 saves almost nothing on the cloudy day, while the optimum still saves about 54 euro. Where does the difference come from?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE9" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-09/"
               "NB09_energy_optimisation.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Wherever energy can be stored or a demand can wait, the same linear program finds the cheapest plan. The function of this section schedules a store of any kind against a price after one change of a name.

The systems and their numbers are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "water_tower"
# choose: home_battery, water_tower, cold_store, heat_pump, bus_depot, data_centre
APPS = {  # (description, capacity in kWh, power limit in kW, efficiency, demand in kWh per hour)
    "home_battery": ("a home battery of 10 kWh for a family", 10, 5, 0.95, 0.4 + 0.6 * (HOURS >= 17) * (HOURS < 22)),
    "water_tower": ("the pumps of a water tower that fill it ahead of the demand", 800, 120, 1.0,
                    40 + 30 * np.sin(np.pi * HOURS / 24)),
    "cold_store": ("a cold store whose walls hold the cold", 300, 60, 0.9, 25 + 10 * (HOURS >= 10) * (HOURS < 18)),
    "heat_pump": ("the heating of a school with a hot water tank", 400, 90, 0.97, 30 + 25 * (HOURS >= 6) * (HOURS < 16)),
    "bus_depot": ("an electric bus depot with a stationary battery", 1500, 400, 0.95,
                  80 + 220 * ((HOURS < 5) | (HOURS >= 22))),
    "data_centre": ("the cooling of a data centre with an ice store", 2000, 500, 1.0, 300 + 50 * np.cos(np.pi * HOURS / 12))}
desc, cap, p_max, eta, demand = APPS[APP]

def store_plan(demand, price, cap, p_max, eta):
    """The cheapest purchases b_t that cover the demand through a store: e_t = e_(t-1) + eta b_t - demand_t."""
    T = len(demand)
    A_eq = np.zeros((T, 2 * T))
    for t in range(T):
        A_eq[t, t] = eta                        # the purchase of hour t
        A_eq[t, T + t] = -1                     # the level after hour t
        if t > 0:
            A_eq[t, T + t - 1] = 1
    b_eq = demand - np.r_[cap / 2, np.zeros(T - 1)]
    bounds = [(0, p_max)] * T + [(0, cap)] * (T - 1) + [(cap / 2, cap)]
    res = linprog(np.r_[price, np.zeros(T)], A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    return res.x[:T], res.fun

buy, cost = store_plan(np.asarray(demand, float), PRICE, cap, p_max, eta)
plain = float(PRICE @ (np.asarray(demand, float) / eta))
print(f"{APP}: {desc}")
print(f"  buying as needed: {plain:.2f} | optimal plan: {cost:.2f} | saving {100 * (1 - cost / plain):.1f} percent")
print("  hours of the largest purchases:", sorted(np.argsort(buy)[-5:].tolist()))

**Python note.** `np.r_[...]` joins numbers and arrays into one array. The store starts half full and must end at least half full, so the plan cannot borrow from the next day.

**Observe and reflect.** For which system does the plan save most? What limits the saving: The capacity, the power or the losses?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Wrap `schedule` in a class that adds the constraints one by one with named methods, and test it against the function of Section 3. |
| Embedded systems | The controller of the battery container cannot run `linprog`. Write a rule with three thresholds that comes within 1 percent of the optimum on both days. |
| Internet of Things | The prices of the next day arrive at 14:00 over the network. Plan the night at 14:00 and compare it with a plan made at midnight. |
| Computer networks | The car park loses its connection to the controller for two hours. Which schedule should each charger fall back to, and what does it cost? |
| Control engineering | Repeat the optimisation every hour with the actual import so far, as model predictive control does, and compare it with the plan of the morning. |
| Data science | Add a random error of 10 percent to the forecast of the solar plant and measure how much of the saving survives. |
| Cybersecurity | An attacker falsifies the price of three evening hours. Which plans change, and how could the controller detect the attack? |
| Artificial intelligence | Train a small regression on 100 optimised days that predicts the power of the battery from the hour, the price and the import, and compare it with the optimum. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-09/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-09).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>